# Exploring results

LLMDyno ships the results of its first study: 15 model configurations (local GGUF models on an RTX 5080 and a cloud model), five agentic coding tasks, the Pi harness, judged by two LLMs. This notebook looks at them the way you would look at your own. It needs `pip install "llmdyno[data,plot]"`.

In [ ]:
import tempfile

import llmdyno

# A throwaway data folder, so this notebook never touches yours. Use llmdyno.init() (no argument) for your own.
llmdyno.init(tempfile.mkdtemp(prefix="llmdyno-notebook-"))
plan_id = llmdyno.load_sample()          # 15 configurations measured on an RTX 5080, bundled with LLMDyno
results = llmdyno.results(plan_id)
results

## As a table

Every table of the Results page is a data frame: `configs` (the leaderboard), `runs`, `cells` (configuration × benchmark), `reviews`, `rubric`.

In [ ]:
results.to_pandas()[["rank", "profile", "final", "correctness", "quality", "wall_s_total", "peak_vram_mib"]].head(8)

In [ ]:
cells = results.to_pandas("cells")
cells.pivot_table(index="profile", columns="benchmark_id", values="correctness").round(2)

## The dashboard's charts

In [ ]:
results.plot.leaderboard();

In [ ]:
results.plot.heatmap("correctness");

In [ ]:
results.plot.pareto(x="wall_s_total");

In [ ]:
results.plot.radar(top=4);

## Weighting what matters to you

The final score weighs correctness, instruction following, code quality and efficiency (50/15/25/10 by default). Change the weights and the ranking follows:

In [ ]:
strict = llmdyno.results(plan_id, correctness=80, instruction_following=10, quality=10, efficiency=0)
[(r["rank"], r["profile"], r["final"]) for r in strict.leaderboard()[:5]]

## Do the judges agree?

In [ ]:
pairs = results.agreement()["pairs"]
[(p["a_name"], p["b_name"], p["n"], round(p.get("pearson") or 0, 2)) for p in pairs]